In [1]:
%pip install --upgrade pip

# Uninstall conflicting packages (including packages that depend on old langchain)
%pip uninstall -y langchain-community langchain-text-splitters langchain-openai langchain-core langsmith beautifulsoup4 python-dotenv langchain-chroma chromadb langchain langchain-together ragas langmem

# Install compatible versions of langchain-core and langchain-openai
%pip install langchain-community==0.4.1
%pip install langchain-text-splitters==1.0.0
%pip install langchain-openai==1.1.0
%pip install langsmith==0.4.49
%pip install langchain==1.1.0

# Install remaining packages
%pip install langchain-chroma==1.0.0
%pip install chromadb==1.3.5
%pip install beautifulsoup4==4.14.2
%pip install python-dotenv==1.2.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.3.1
    Uninstalling pip-24.3.1:
      Successfully uninstalled pip-24.3.1
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached pyyaml-6.0.3-cp310-cp310-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached numpy-2.2.6-cp310-cp310-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Usin

In [2]:
# New OS parameter to avoid warnings.  
# This will not have a material impact on your code, but prevents warnings from appearing related to new LangChain features.
import os
os.environ['USER_AGENT'] = 'RAGUserAgent'

In [3]:
from langchain_community.document_loaders import WebBaseLoader
import bs4
import openai
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langsmith import Client
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

In [4]:
# OpenAI Setup
_ = load_dotenv(dotenv_path='env.txt')
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
openai.api_key = os.environ['OPENAI_API_KEY']

In [5]:
#### INDEXING ####

In [6]:
# Load Documents
loader = WebBaseLoader(
    web_paths=("https://kbourne.github.io/chapter1.html",), 
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

In [7]:
# Split
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)
splits = text_splitter.split_documents(docs)

In [8]:
# Embed
vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=OpenAIEmbeddings()
)

retriever = vectorstore.as_retriever()

In [11]:
#### RETRIEVAL and GENERATION ####

query = "How does RAG compare with fine-tuning?"
relevant_docs = retriever.invoke(query)
relevant_docs

[Document(id='669a7b16-3199-44c7-a906-946401df30e6', metadata={'source': 'https://kbourne.github.io/chapter1.html'}, page_content="Ultimately, when deciding between RAG and fine-tuning, consider your specific use case and requirements. RAG is generally superior for retrieving factual information that is not present in the LLM's training data or is private. It allows for dynamic integration of external knowledge without modifying the model's weights. Fine-tuning, on the other hand, is more suitable for teaching the model specialized tasks or adapting it to a specific domain. Keep in mind the limitations of context window sizes and the potential for overfitting when fine-tuning on a specific dataset."),
 Document(id='ffea2adb-4066-4eb8-ad40-51e3d246affc', metadata={'source': 'https://kbourne.github.io/chapter1.html'}, page_content="Even smaller companies are not able to access much of their internal data resources very effectively.  Larger companies are swimming in petabytes of data that

In [12]:
# Prompt
client = Client()
prompt = client.pull_prompt("jclemens24/rag-prompt")
print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'jclemens24', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '1a1f3ccb9a5a92363310e3b130843dfb2540239366ebe712ddd94982acc06734'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


In [13]:
# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [14]:
# LLM
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

In [15]:
# Chain it all together with LangChain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [17]:
# Question - run the chain
print(rag_chain.invoke("What are the advantages of using RAG?"))

The advantages of using RAG (Retrieval-Augmented Generation) include:

1. **Improved Accuracy and Relevance**: RAG enhances the accuracy and relevance of responses by incorporating specific information from a database or dataset in real time, ensuring outputs are based on both the model’s pre-existing knowledge and the most current data.

2. **Customization and Flexibility**: RAG allows for tailored responses based on domain-specific needs by integrating a company's internal databases, creating personalized experiences and detailed outputs.

3. **Expanding Knowledge**: RAG helps expand the model’s knowledge beyond its training data, making it more useful for organizations that need to connect to data the model is not familiar with.

4. **Product Recommendations**: It can enhance e-commerce product recommendation engines and generate personalized descriptions based on customer behavior.

5. **Training and Education**: RAG can customize learning materials for educational organizations an